# **Notebook 1: Exploratory Data Analysis**
**Goal:** Load images and annotations, visualize ground truth overlays, understand data distribution.

It will tell you:
- What tube lids actually look like from overhead
- How the angle annotation is defined visually
- How many tubes per image, background variety, etc.

In [ ]:
import os, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
import gdown
DIST_THRESH  = 30.0

DATA_ROOT = '/content/dataset'
IMG_DIR   = os.path.join(DATA_ROOT, 'images')
ANN_CSV   = os.path.join(DATA_ROOT, 'annotations.csv') # Updated to local path
os.makedirs(IMG_DIR, exist_ok=True)

# 1. Download the CSV if it doesn't exist
CSV_ID = '1V2ZuaDBe3pZTBRallIWLVHtpCDXzq9jK'
if not os.path.exists(ANN_CSV):
    gdown.download(id=CSV_ID, output=ANN_CSV, quiet=False)

# 2. Load annotations
ann = pd.read_csv(ANN_CSV)

print(f"Total annotations : {len(ann)}")
print(f"Unique images     : {ann['image'].nunique()}")
print("\n--- CSV Head ---")
print(ann.head())

In [ ]:
# ── Distribution: tubes per image ───────────────────────────────────────────
tubes_per_image = ann.groupby('image').size()
tubes_per_image.value_counts().sort_index().plot(kind='bar', title='Tubes per image')
plt.xlabel('# tubes')
plt.ylabel('# images')
plt.tight_layout()
plt.show()
print(tubes_per_image.describe())
# ── Distribution: angle_deg ──────────────────────────────────────────────────
plt.figure(figsize=(8,3))
plt.hist(ann['angle_deg'], bins=36, edgecolor='k')
plt.title('Distribution of tube lid angles (0–360°)')
plt.xlabel('angle_deg')
plt.ylabel('count')
plt.tight_layout()
plt.show()
# ── Helper: draw one image with GT overlays ──────────────────────────────────
def draw_gt(image_name, ann_df, img_dir, arrow_len=30):
    img_path = os.path.join(img_dir, image_name)
    img = np.array(Image.open(img_path).convert('RGB'))
    rows = ann_df[ann_df['image'] == image_name]

    fig, ax = plt.subplots(1, 1, figsize=(10, 7))
    ax.imshow(img)

    for _, row in rows.iterrows():
        cx, cy = row['center_x'], row['center_y']
        angle_rad = math.radians(row['angle_deg'])

        # Draw bounding box
        rect = mpatches.Rectangle(
            (row['bbox_x'], row['bbox_y']),
            row['bbox_w'], row['bbox_h'],
            linewidth=1.5, edgecolor='lime', facecolor='none',
            label='bbox'
        )
        ax.add_patch(rect)

        # Draw center
        ax.plot(cx, cy, 'ro', markersize=5)

        # Draw angle arrow (joint→tab direction)
        # angle=0 → +X direction (rightward), CCW positive
        dx = arrow_len * math.cos(angle_rad)
        dy = -arrow_len * math.sin(angle_rad)   # Y increases downward in image coords
        ax.annotate('', xy=(cx+dx, cy+dy), xytext=(cx, cy),
                    arrowprops=dict(arrowstyle='->', color='yellow', lw=2))

        ax.text(cx+2, cy-12, f"{row['angle_deg']:.0f}°",
                color='yellow', fontsize=8, fontweight='bold')

    ax.set_title(image_name)
    ax.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
import zipfile
import os

# 1. Setup paths
ZIP_PATH = '/content/images.zip' # Make sure this matches the name of your uploaded file
DATA_ROOT = '/content/dataset'
FINAL_CSV    = '/content/final_predictions.csv'
METRICS_JSON = '/content/metrics.json'

IMG_DIR = os.path.join(DATA_ROOT, 'images')

os.makedirs(IMG_DIR, exist_ok=True)

# 2. Extract the ZIP
print(f"Extracting {ZIP_PATH}...")
try:
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DATA_ROOT)
    print("Extraction complete!")

    # Quick check
    extracted_files = len(os.listdir(IMG_DIR))
    print(f"Total images now in {IMG_DIR}: {extracted_files}")
except FileNotFoundError:
    print(f"Error: Could not find '{ZIP_PATH}'. Did you upload it and name it correctly?")

In [ ]:
# Visualize first 6 images
sample_images = ann['image'].unique()[:6]
for img_name in sample_images:
    draw_gt(img_name, ann, IMG_DIR)
# ── Inspect a single cropped tube lid ────────────────────────────────────────
# Pick one annotation and crop around its bounding box to see the tab clearly
sample_row = ann.iloc[0]
img = Image.open(os.path.join(IMG_DIR, sample_row['image'])).convert('RGB')
pad = 10
x1 = max(0, int(sample_row['bbox_x']) - pad)
y1 = max(0, int(sample_row['bbox_y']) - pad)
x2 = int(sample_row['bbox_x'] + sample_row['bbox_w']) + pad
y2 = int(sample_row['bbox_y'] + sample_row['bbox_h']) + pad
crop = img.crop((x1, y1, x2, y2))

plt.figure(figsize=(4,4))
plt.imshow(crop)
plt.title(f"Cropped lid — angle: {sample_row['angle_deg']:.1f}°")
plt.axis('off')
plt.show()

print("Study this crop: locate the joint and the tab visually.")
print("This will inform which angle estimation method to use.")
# ── Summary stats ─────────────────────────────────────────────────────────────
print("=== Annotation Summary ===")
print(f"Total tubes            : {len(ann)}")
print(f"Images                 : {ann['image'].nunique()}")
print(f"Tubes per image (mean) : {len(ann)/ann['image'].nunique():.2f}")
print(f"bbox_w range           : {ann['bbox_w'].min():.1f} – {ann['bbox_w'].max():.1f} px")
print(f"bbox_h range           : {ann['bbox_h'].min():.1f} – {ann['bbox_h'].max():.1f} px")
print(f"angle_deg range        : {ann['angle_deg'].min():.1f} – {ann['angle_deg'].max():.1f}")